# 05 — Tool Use

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 30 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=40-DUQ4Cf2Q)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A calculator capability with an inspectable schema, registry metadata, agent assignment, successful execution, and a sanitized error path. You will also register an externally described JSON-schema tool with `Agent.add_tool_spec()`.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(5, 'Tool Use')


## Prerequisites and setup

**Course prerequisites:** Complete `course-01-hello-world`.

**Execution requirements:** Offline. No provider key is needed because this lesson exercises the tool contracts directly; lesson 09 connects the same contracts to ModelRuntime.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Describe a tool as a typed, named capability.
- Inspect JSON-schema-compatible parameters and registry metadata.
- Share and assign tools to agents.
- Handle validation, duplicate registration, and execution errors.


## Mental model

A **tool** is a capability with a name, description, argument schema, execution handler, and policy metadata. Registration makes it discoverable. Assignment makes it available to an Agent. A model may later request a tool call, but the handler is still ordinary application code whose inputs and failures must be validated.


In [ ]:
show_flow(
('Schema', 'arguments and policy', 'tool'),
('Registry', 'discover and share', 'reef'),
('Agent', 'available capability', 'agent'),
('Result', 'data or error', 'spore'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Register a typed shared tool

The decorator reads Python type hints into metadata. Resetting the registry makes this notebook safe to run again in the same kernel.


In [ ]:
from praval import Agent, agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import list_tools, tool

reset_reef()
reset_tool_registry()
tool_results = []
tool_spores = []


@tool(
    "course_calculator",
    description="Multiply two numbers for the tool course.",
    category="education", shared=True,
)
def multiply(a: float, b: float) -> float:
    stage("tool", "executed", f"{a} × {b}")
    if abs(a) > 1_000_000 or abs(b) > 1_000_000:
        raise ValueError("input exceeds the lesson limit")
    return a * b


### What just happened?

`multiply` remains callable, and `_praval_tool` now carries its schema and execution wrapper. `shared=True` makes it discoverable to every agent.

### Why this matters

A typed contract gives providers, operators, and tests the same view of the capability.


In [ ]:
registered = [
    item for item in list_tools() if item["tool_name"] == "course_calculator"
]
assert len(registered) == 1
show_json(registered[0], "Registered tool metadata")


### Attach the tool to a message-driven agent

The agent declares its tool by registry name. The handler uses the registered function and publishes a result Spore.


In [ ]:
@agent(
    "course-tool-agent", responds_to=["calculate"],
    tools=["course_calculator"], auto_broadcast=False,
)
def calculator_agent(spore):
    stage("tool-agent", "arguments received", str(spore.knowledge), spore)
    result = multiply(spore.knowledge["a"], spore.knowledge["b"])
    broadcast({"type": "calculation_result", "result": result})


@agent(
    "course-tool-output", responds_to=["calculation_result"],
    auto_broadcast=False,
)
def calculator_output(spore):
    tool_spores.append(spore)
    tool_results.append(spore.knowledge["result"])
    stage("output", "tool result received", str(spore.knowledge["result"]), spore)


assert calculator_agent.has_tool("course_calculator")


### What just happened?

The decorator resolved the shared registry entry and exposed it through the agent's tool collection.

### Why this matters

Tool ownership and sharing are separate from the message route that initiates work.


### Execute and inspect the successful path

The initial Spore supplies arguments. The output agent receives a new typed result message after the tool executes.


In [ ]:
start_agents(
    calculator_agent,
    calculator_output,
    initial_data={"type": "calculate", "a": 7, "b": 6},
    channel="course-tools",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert tool_results == [42]


### What just happened?

The registered Python handler returned `42`; the agent converted it into a workflow result that another participant could consume.

### Why this matters

Tools create controlled side-effect and computation boundaries inside agent systems.


In [ ]:
show_spore(tool_spores[0], "Tool result Spore")
show_timeline()


### See a tool error as a tool error

`execute_as_tool` wraps handler exceptions with the registered tool identity. The example checks the category without exposing secrets or a traceback.


In [ ]:
from praval.core.exceptions import ToolError

try:
    multiply.execute_as_tool(2_000_000, 2)
except ToolError as error:
    safe_error = type(error).__name__ + ": input rejected by tool policy"
else:
    raise AssertionError("Expected the tool policy to reject oversized input")

show_callout("Expected tool error", safe_error, role="tool")


### What just happened?

The failure stayed at the tool boundary and remained attributable to `course_calculator`.

### Why this matters

Structured tool failures let an agent or human decide whether to correct, retry, or stop without mistaking the failure for model prose.


### Register an external JSON-schema tool

`add_tool_spec` supports tools discovered outside a decorator, including MCP. The schema is validated and duplicate names are rejected.


In [ ]:
from praval.models import ToolSpec

external_agent = Agent(
    "course-external-tool-agent",
    provider="notebook-lifecycle", model="notebook-lifecycle-model",
)
lookup_spec = ToolSpec(
    name="course_lookup",
    description="Look up one lesson topic.",
    parameters={
        "type": "object",
        "properties": {"topic": {"type": "string"}},
        "required": ["topic"],
    },
    metadata={"category": "education"},
)
external_agent.add_tool_spec(lookup_spec, lambda topic: {"topic": topic})
assert "course_lookup" in external_agent.tools

try:
    external_agent.add_tool_spec(lookup_spec, lambda topic: topic)
except ValueError as error:
    assert "already registered" in str(error)


### What just happened?

The Agent now has an externally defined tool while preserving one tool registry and one provider-neutral schema format.

### Why this matters

External protocols should adapt to the framework's tool contract instead of creating parallel execution paths.


In [ ]:
show_json(external_agent.tools["course_lookup"], "External tool on Agent")


## Your turn

Add a shared `course_divide` tool with typed parameters. Reject division by zero, put it in the `education` category, and inspect it with `list_tools`.


In [ ]:
# @tool("course_divide", category="education", shared=True)
# def divide(numerator: float, denominator: float) -> float:
#     if denominator == 0:
#         raise ValueError("denominator cannot be zero")
#     return numerator / denominator


## Common mistake

**Mistake:** Treating a tool description as input validation.

**Correction:** Validate in the schema and handler. Descriptions guide models and humans but do not enforce application invariants.


<details>
<summary>Under the hood</summary>

Provider adapters serialize `ToolSpec` into native provider formats. Runtime tool calls are mapped back to the registered handler, approval policy is checked, and the handler result becomes a provider-neutral `ToolResult`.

</details>


## Recap

- Tools combine schema, handler, discovery metadata, and policy.
- Typed decorators make local capabilities easy to register.
- `add_tool_spec` integrates external schemas without another registry.
- Execution errors should remain structured and attributable.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
external_agent.close()
calculator_agent._praval_agent.close()
calculator_output._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()


### Next lesson

Continue to `06_agent_memory.ipynb` to give an Agent typed memories that can be stored, inspected, recalled, and cleared.
